# Lyrics Text Analysis

## Author: 
Yangshunshun Lu
## AI Disclosure: 
This code uses Gemini for debugging and learning TextBlob
## Description:
This script performs Natural Language Processing (NLP) feature extraction on song lyrics by applying Part-of-Speech (POS) filtering, lemmatization, and multi-lingual stopword removal. It then calculates and appends the text sentiment polarity score for the purified lyrics using TextBlob.

## Set up

In [ ]:
import pandas as pd
import os
import numpy as np
import spacy
from nltk.corpus import stopwords
from textblob import TextBlob

# Load the spaCy English model (keeping the tagger active for POS)
nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Combine stopwords from multiple languages and add custom filler words
stop_words_en = set(stopwords.words('english'))
stop_words_de = set(stopwords.words('german'))
stop_words_fr = set(stopwords.words('french'))
stop_words_es = set(stopwords.words('spanish'))

custom_stops = {
    'get', 'go', 'know', 'like', 'let', 'wanna', 'gotta', 'gonna',
    'yeah', 'oh', 'ooh', 'ah', 'uh', 'la', 'da', 'mm', 'lo'
}

# Ultimate stopword list
final_stops = stop_words_en | stop_words_de | stop_words_fr | stop_words_es | custom_stops

# Define allowed Part-of-Speech (POS) tags to retain meaningful words only
ALLOWED_POS = {'NOUN', 'PROPN', 'ADJ', 'VERB', 'ADV'}

## Preprocessing using POS
only keep the adv, adj, verb, noun which are meaningful terms in the lyrics

In [8]:
print("1. Loading base English dataset...")
input_file = '../../Data/cleaned_data/lyricsFeatures_mergeERUS_clean_enonly_token.csv'
df = pd.read_csv(input_file)

# Fill missing values
df['lyrics'] = df['lyrics'].fillna('')
df['processed_lyrics'] = df['processed_lyrics'].fillna('')

# --- POS Filtering ---
print("2. Applying deep POS filtering...")

def pos_filter_lyrics(text):
    """
    Filters and lemmatizes text based on allowed Parts of Speech (POS) and stopwords.
    
    This function cleans the input text by converting it to lowercase and removing 
    bracketed structural tags (e.g., [Chorus]). It then processes the text using 
    a globally defined NLP model (`nlp_en`) to extract token lemmas. Tokens are 
    retained only if they match the allowed POS tags (`ALLOWED_POS`), are not in 
    the custom stopword list (`final_stops`), and are strictly longer than one character.

    Args:
        text (str): The raw lyrics text to be processed.

    Returns:
        str: A space-separated string of the filtered and lemmatized tokens. 
             Returns an empty string if the input is not a valid string or is empty.
    """
    if not isinstance(text, str) or len(text.strip()) == 0: 
        return ""
    
    # Pre-cleaning: lowercase and remove content within brackets (e.g., [Chorus])
    clean_text = text.lower().replace(r'\[.*?\]', ' ')
    doc = nlp_en(clean_text)
    
    # Filter tokens based on POS, stopword list, and length
    tokens = [
        t.lemma_ for t in doc 
        if t.pos_ in ALLOWED_POS and t.text not in final_stops and len(t.text) > 1
    ]
    return " ".join(tokens)

# Apply the filter to create the highly purified lyrics column
df['pos_filtered_lyrics'] = df['processed_lyrics'].apply(pos_filter_lyrics)


1. Loading base English dataset...
2. Applying deep POS filtering...


## Text Sentiment Calculation

In [9]:
# --- Calculate Lyrics Sentiment ---
print("3. Calculating textsentiment scores...")

df['text_sentiment'] = df['pos_filtered_lyrics'].astype(str).apply(lambda x: TextBlob(x).sentiment.polarity)

print("Data preprocessing and feature extraction completed.")

3. Calculating textsentiment scores...
Data preprocessing and feature extraction completed.


## Save Result

In [ ]:
model_path = '../../Data/cleaned_data/lyricsFeatures_mergeERUS_clean_enonly_token_pos_ldaTfidf.csv'
df.to_csv(model_path, index=False, encoding='utf-8-sig')

print(f"\n All pipelines executed successfully! The final dataset has been saved to: {model_path}")


✨ All pipelines executed successfully! The final dataset has been saved to: ../../Data/cleaned_data/lyricsFeatures_mergeERUS_clean_enonly_token_pos_ldaTfidf.csv


little check

In [11]:
df.head(5)

,country,date,rank,uri,artist_name,track_name,source,peak_rank,previous_rank,days_on_chart,...,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,processed_lyrics,pos_filtered_lyrics,text_sentiment
0,Austria,2017-01-01,4,spotify:track:76gwG061wqdatYhicO2hG4,Alle Farben,Bad Ideas,b1,4,-1,1,...,0.710,-6.630,0.0331,0.0163,0.023700,0.0892,0.678,lie face lose track every tongue taste like wh...,lie face lose track tongue taste whole world u...,0.061429
1,Austria,2017-01-01,9,spotify:track:5uCax9HTNlzGybIStD3vDh,James Arthur,Say You Won't Let Go,Columbia,9,-1,1,...,0.557,-7.398,0.0590,0.6950,0.000000,0.0902,0.494,meet dark light make feel though enough dance ...,meet dark light make feel enough dance night a...,0.182788
2,Austria,2017-01-01,51,spotify:track:3CRDbSIZ4r5MsZ0YwxuEkn,Twenty One Pilots,Stressed Out,Fueled By Ramen,51,-1,1,...,0.637,-5.677,0.1410,0.0462,0.000023,0.0602,0.648,wish find well sound one ever hear wish well v...,wish find well sound ever hear wish voice sing...,0.258843
3,Austria,2017-01-01,59,spotify:track:550rQQCGkrTzvp4SfpOPzx,Pentatonix,Hallelujah,RCA Records Label,59,-1,1,...,0.377,-7.385,0.0356,0.4540,0.000000,0.3380,0.366,\n hear secret chord david play please lord r...,hear secret chord david play lord really care ...,0.048516
4,Austria,2017-01-01,127,spotify:track:7129iqBafaphfc3WPCGC0L,Lukas Graham,7 Years,Warner Records,127,-1,1,...,0.479,-5.830,0.0551,0.2900,0.000000,0.3910,0.346,seven year old mama tell \n go make friend lo...,year old mama tell make friend lonely year old...,0.046545
